## Trade with ChatGPT

In [ ]:
import sys
sys.path.append("D:\python_project\FinRL-Meta\FinRL-Meta")  # Your FinRL-Meta dir
import pandas as pd
from meta.data_processors.akshare import Akshare # https://github.com/AI4Finance-Foundation/FinRL-meta
import datetime
import matplotlib.pyplot as plt

#### Read data

In [ ]:
df = pd.read_csv("./data/maotai.csv")
df.head(1)

#### Generate Signal from ChatGPT

In [ ]:
def generate_signal_from_chatgpt(x):
    chatgpt_res = x.chatgpt_res
    if "持有不动" in chatgpt_res:
        signal = 0
    if "小幅加仓" in chatgpt_res:
        signal = 1
    elif "小幅减仓" in chatgpt_res:
        signal = -1
    if "大幅加仓" in chatgpt_res:
        signal = 2
    elif "大幅减仓" in chatgpt_res:
        signal = -2
    return signal

In [ ]:
df["signal"] = df.apply(generate_signal_from_chatgpt,axis = 1)

In [ ]:
df["signal"]

#### Generate Signal by yourself
Here the result of ChatGPT is just a suggestion, you need to make trading signal after considering both news and the result of ChatGPT

In [ ]:
def generate_signal_from_chatgpt_by_yourself(x):
    chatgpt_res = x.chatgpt_res
    news = x.text_a
    signal = input(
        f'- {news}\n - {chatgpt_res}\n (Please input 1 for buy or 0 for hold or -1 for sell)'
        )
    return int(signal)

In [ ]:
df["signal_by_yourself"] = df.apply(generate_signal_from_chatgpt_by_yourself,axis = 1)

In [ ]:
df["signal_by_yourself"]

#### Fetching price data from FinRL-Meta

In [ ]:
df.DATE = pd.to_datetime(df.DATE)
df.DATE = df.DATE.dt.date.astype(str)
max_date = datetime.datetime.strptime(df.DATE.max(),"%Y-%m-%d")
max_date += datetime.timedelta(days=10)
max_date = max_date.strftime("%Y-%m-%d")

In [ ]:
time_interval = "daily"
start_date= df.DATE.min()
end_date = max_date
adjust=""

ticket_list=[f'{df.iloc[0].CODE}.SH', ]

In [ ]:
as_processor = Akshare("akshare",start_date=start_date,end_date=end_date,time_interval=time_interval)
as_processor.download_data(ticket_list)
as_processor.dataframe.shape

#### Trading
Since we only have the news for several days here, we only calculated the dicision results made by Chatgpt or by combining Chatgpt and your own opinion

In [ ]:
price_df = as_processor.dataframe
price_df = price_df.reset_index(drop=True)
price_df.head(1)

In [ ]:
def cal_reward(x,by_yourself = False, hold = False):
    # get price
    this_date = x.DATE
    this_price = price_df[price_df.time == this_date]
    
    while this_price.shape[0] == 0:
        this_date = datetime.datetime.strptime(this_date,"%Y-%m-%d")
        this_date += datetime.timedelta(days=1)
        this_date = this_date.strftime("%Y-%m-%d")
        this_price = price_df[price_df.time == this_date]
    
    next_price = price_df.iloc[this_price.index+1]
    this_close = this_price.close.item()
    next_open = next_price.open.item()
    next_close = next_price.close.item() # T+1
    
    if hold:
        signal = 1
    else:
        if by_yourself:
            signal = x.signal_by_yourself
        else:
            signal = x.signal

    reward = signal * (next_close-this_close)/this_close
    return reward


In [ ]:
df["reward"] = df.apply(lambda x:cal_reward(x),axis = 1)
df["reward_by_yourself"] = df.apply(lambda x:cal_reward(x,by_yourself = True),axis = 1)
df["reward_hold"] = df.apply(lambda x:cal_reward(x,hold = True),axis = 1)

In [ ]:
reward_list = (df["reward"] +1 ).cumprod().to_list()
reward_hold_list = (df["reward_hold"] +1 ).cumprod().to_list()
reward_yourself_list = (df["reward_by_yourself"] +1 ).cumprod().to_list()
reward_list = [i/reward_list[0] for i in reward_list]
reward_hold_list = [i/reward_hold_list[0] for i in reward_hold_list]
reward_yourself_list = [i/reward_yourself_list[0] for i in reward_yourself_list]


In [ ]:
date_list = df.DATE
plt.figure(figsize=(20,5))
plt.plot(date_list, reward_list, label = "Reward by ChatGPT")
plt.plot(date_list, reward_yourself_list, label= "Reward with ChatGPT")
plt.plot(date_list, reward_hold_list, label= "Buy and hold")
plt.title("Maotai ChatGPT Trading Results")
plt.legend()
plt.show()